In [6]:
import pandas as pd

### Load Dataset

In [7]:
df = pd.read_csv("../data/email_evaluation_dataset_isha-bhole.csv")
df.head()

,id,email_text,expected_action,expected_tone
0,1,Newsletter: Top ML papers this week.,ignore,neutral
1,2,Weekly newsletter: Latest AI research updates.,ignore,neutral
2,3,Automatic reply: Out of office till 2025-12-20.,ignore,neutral
3,4,Scholarship application status: Shortlisted. N...,ignore,neutral
4,5,"Payment reminder: INR INR 10,602 outstanding. ...",notify,polite


###  Create a cleaned text column
Before applying or analysing rules, a clean_text column is created from email_text by lowercasing and removing non‑alphabet characters.

In [8]:
df["clean_text"] = (
    df["email_text"]
      .astype(str)
      .str.lower()
      .str.replace("[^a-zA-Z ]", "", regex=True)
)
df.head()

,id,email_text,expected_action,expected_tone,clean_text
0,1,Newsletter: Top ML papers this week.,ignore,neutral,newsletter top ml papers this week
1,2,Weekly newsletter: Latest AI research updates.,ignore,neutral,weekly newsletter latest ai research updates
2,3,Automatic reply: Out of office till 2025-12-20.,ignore,neutral,automatic reply out of office till
3,4,Scholarship application status: Shortlisted. N...,ignore,neutral,scholarship application status shortlisted nex...
4,5,"Payment reminder: INR INR 10,602 outstanding. ...",notify,polite,payment reminder inr inr outstanding due jan


### Define the email_assistant logic
Here a function email_assistant(email_text) is defined that reads the email text, converts it to lowercase, and looks for simple keywords.
Based on these keywords it returns a pair (action, tone), for example ("respond", "urgent") for deadlines or payments, and ("ignore", "neutral") for marketing emails.

In [10]:
def email_assistant(email_text):
    text = str(email_text).lower()
    if any(k in text for k in ["urgent", "deadline", "submit", "due", "payment", "invoice", "overdue"]):
        return "respond", "urgent"
    if any(k in text for k in ["suspicious", "security alert", "phishing", "account suspended"]):
        return "notify", "urgent"
    if any(k in text for k in ["sale", "offer", "discount", "free", "lottery", "win"]):
        return "ignore", "neutral"
    if any(k in text for k in ["mentor", "professor", "meeting", "project", "thesis", "assignment"]):
        return "respond", "polite"
    return "respond", "neutral"

### Generate predictions for every email
The output is split into two new columns, predicted_action and predicted_tone, which represent what the rule‑based assistant would do for each email.

In [11]:
df["predicted_action"], df["predicted_tone"] = zip(
    *df["email_text"].apply(email_assistant)
)

df[["email_text", "expected_action", "predicted_action",
    "expected_tone", "predicted_tone"]].head()

,email_text,expected_action,predicted_action,expected_tone,predicted_tone
0,Newsletter: Top ML papers this week.,ignore,respond,neutral,neutral
1,Weekly newsletter: Latest AI research updates.,ignore,respond,neutral,neutral
2,Automatic reply: Out of office till 2025-12-20.,ignore,respond,neutral,neutral
3,Scholarship application status: Shortlisted. N...,ignore,respond,neutral,neutral
4,"Payment reminder: INR INR 10,602 outstanding. ...",notify,respond,polite,urgent


### Mark which predictions are correct
Two Boolean columns, action_correct and tone_correct, are created by comparing predictions with the expected labels.
For each row these columns are True when the assistant’s output matches the human label, and False when it is wrong.

In [12]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]

df[["expected_action", "predicted_action", "action_correct"]].head()
df[["expected_tone", "predicted_tone", "tone_correct"]].head()

,expected_tone,predicted_tone,tone_correct
0,neutral,neutral,True
1,neutral,neutral,True
2,neutral,neutral,True
3,neutral,neutral,True
4,polite,urgent,False


### Calculate overall accuracy
This tells how often the assistant made the right decision across the whole evaluation dataset.

In [13]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

print("Action accuracy (%):", action_accuracy)
print("Tone accuracy (%):", tone_accuracy)

Action accuracy (%): 56.99999999999999
Tone accuracy (%): 67.0


### Error analysis
Rows where action_correct or tone_correct are False are filtered into separate tables for mistakes.

In [21]:
errors = df[df["action_correct"] == False][
    ["email_text", "expected_action", "predicted_action"]
]
num_action_errors = len(errors)
print("Number of action prediction errors:", num_action_errors)
errors.head(20)

Number of action prediction errors: 43


,email_text,expected_action,predicted_action
0,Newsletter: Top ML papers this week.,ignore,respond
1,Weekly newsletter: Latest AI research updates.,ignore,respond
2,Automatic reply: Out of office till 2025-12-20.,ignore,respond
3,Scholarship application status: Shortlisted. N...,ignore,respond
4,"Payment reminder: INR INR 10,602 outstanding. ...",notify,respond
6,Server maintenance completed. All systems normal.,ignore,respond
7,Thesis review meeting confirmed for Jan 5. Pre...,notify,respond
12,Thesis review meeting confirmed for 2025-12-25...,notify,respond
16,Project extension approved till Jan 5. Update ...,ignore,respond
17,Could you clarify the dataset format for crop ...,ignore,respond


In [22]:
tone_errors = df[df["tone_correct"] == False][
    ["email_text", "expected_tone", "predicted_tone"]
]
num_tone_errors = len(tone_errors)
print("Number of tone prediction errors:", num_tone_errors)
tone_errors.head(20)

Number of tone prediction errors: 33


,email_text,expected_tone,predicted_tone
4,"Payment reminder: INR INR 10,602 outstanding. ...",polite,urgent
10,Quick question: Can you review my code changes...,polite,neutral
11,Need help with NLTK stemming. Any recommendati...,polite,neutral
13,Reminder: Team meeting tomorrow at 10 AM. Plea...,urgent,polite
14,Quick question: Can you review my code changes...,polite,neutral
16,Project extension approved till Jan 5. Update ...,neutral,polite
18,Urgent: Meeting rescheduled to today 3 PM. Joi...,polite,urgent
20,Reminder: Team meeting tomorrow at 3 PM IST. P...,urgent,polite
23,Project sync scheduled for 2025-12-25 at 4 PM....,neutral,polite
24,Grades updated for ML course. Check portal for...,polite,neutral


### Save the evaluated dataset

In [23]:
out_path = "../data/milestone2_output_isha-bhole.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: ../data/milestone2_output_isha-bhole.csv


### Reflection

# 1. Which emails were hardest to classify?
The hardest emails were the “in‑between” ones that were partly important but not clearly urgent, such as general project updates, friendly check‑ins, and information emails that sometimes needed a reply and sometimes did not. Many promotional or newsletter‑style emails were also confusing when they mentioned real tasks (like conferences or events) along with marketing language.

# 2. Why did your rules fail there?
The rules failed because they only look for a few fixed keywords and cannot understand full sentence meaning or context. For example, if the text contains words like “meeting” or “deadline” the rule always predicts respond, even when the email is just a reminder or summary and does not really need an action. The rules also cannot understand tone correctly when there is no clear word like “urgent”, so they often guess neutral or polite even when the message feels more serious.

# 3. How could an LLM do better?
An LLM could read the whole email and understand the intent, such as whether the sender is actually asking for a reply or only sharing information. It can also detect subtle signals of urgency and emotion instead of depending only on a small keyword list. This would reduce both action errors and tone errors, especially on borderline cases where simple rules are not enough.